# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [49]:
# Clean consumer transactions
n = 13000
year1 = rng.lognormal(np.log(45), 0.8, n)   # median about $45
year2 = rng.lognormal(np.log(43), 0.8, n)   # median about $43, a little lower

df = pd.DataFrame({
    "customer_id": rng.integers(1, 8001, 2 * n),
    "year": [1] * n + [2] * n,
    "basket_value": np.concatenate([year1, year2]),
})

cancel_chance = np.where(df["basket_value"] > 100, 0.25, 0.05)
df["cancelled"] = rng.random(2 * n) < cancel_chance

print("median:", np.median(df["basket_value"]).round(2), " mean:", df["basket_value"].mean().round(2))
print(df.groupby("year")["cancelled"].sum())

median: 44.09  mean: 61.09
year
1    1041
2    1046
Name: cancelled, dtype: int64


In [50]:
# The truth: the consumer average per year, recorded before contamination
real = df[df["cancelled"] == False]
truth = real.groupby("year")["basket_value"].mean()
true_change = (truth[2] / truth[1] - 1) * 100

print(truth.round(2))
print("True change %:", round(true_change, 2))

year
1    57.83
2    57.07
Name: basket_value, dtype: float64
True change %: -1.33


In [51]:
# Contamination 1: B2B orders
b2b = pd.DataFrame({
    "customer_id": rng.integers(9001, 9061, 120),   # B2B ids are 9001 to 9060
    "year": [1] * 60 + [2] * 60,
    "basket_value": rng.lognormal(np.log(1500), 0.6, 120),   # median about $1,500
    "cancelled": False,
})
df2 = pd.concat([df, b2b])
print("Rows after adding B2B:", len(df2))

Rows after adding B2B: 26120


In [52]:
# Contamination 2: what gets logged changes between year 1 and year 2
dashboard = df2[(df2["year"] == 2) | (df2["cancelled"] == False)]
print("Rows the dashboard sees:", len(dashboard))

Rows the dashboard sees: 25079


### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [53]:
# The dashboard's number vs the truth
naive = dashboard.groupby("year")["basket_value"].mean()
naive_change = (naive[2] / naive[1] - 1) * 100

print(naive.round(2))
print("Dashboard change %:", round(naive_change, 2))
print("Gap (percentage points):", round(naive_change - true_change, 2))
print("Gap in dollars:", (naive - truth).round(2).to_dict())

year
1    67.06
2    69.41
Name: basket_value, dtype: float64
Dashboard change %: 3.51
Gap (percentage points): 4.84
Gap in dollars: {1: 9.22, 2: 12.35}


After adding 60 B2B orders per year and log cancelled orders only in year 2, the dashboard's naive mean rises from 67.06 to 69.41 when the true consumer average fall from 57.83 to 57.07, which overstating growth by 4.84 percentage points.

---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [54]:
# How much of the gap is the skew and the B2B tail?
no_cancel = dashboard[dashboard["cancelled"] == False]
step1 = no_cancel.groupby("year")["basket_value"].mean()
step1_change = (step1[2] / step1[1] - 1) * 100

b2b_part = step1_change - true_change

print(step1.round(2))
print("B2B contribution (percentage points):", round(b2b_part, 2))
print("B2B contribution in dollars:", (step1 - truth).round(2).to_dict())

year
1    67.06
2    66.65
Name: basket_value, dtype: float64
B2B contribution (percentage points): 0.72
B2B contribution in dollars: {1: 9.22, 2: 9.58}


In [55]:
# How much is the change in what was logged?
logging_part = naive_change - step1_change

print("Logging contribution (percentage points):", round(logging_part, 2))
print("Logging contribution in dollars:", (naive - step1).round(2).to_dict())
print("Total:", round(b2b_part + logging_part, 2), "= gap from Phase 1")

Logging contribution (percentage points): 4.12
Logging contribution in dollars: {1: 0.0, 2: 2.76}
Total: 4.84 = gap from Phase 1


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail |0.72 pp of the 4.84 pp gap; +9.22 (Y1), +9.58 (Y2) |Removed cancelled orders, compared with truth (no B2B)|
| Change in what was logged |4.12 pp of the 4.84 pp gap; 0, +2.76 |Compared the dashboard with the same data after removing cancelled orders|

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [56]:
# Median, trimmed mean, and the mean after your B2B exclusion rule
y1 = no_cancel[no_cancel["year"] == 1]["basket_value"]
y2 = no_cancel[no_cancel["year"] == 2]["basket_value"]

print("Median:          ", round(np.median(y1), 2), round(np.median(y2), 2))
print("Trimmed mean:    ", round(stats.trim_mean(y1, 0.1), 2), round(stats.trim_mean(y2, 0.1), 2))
print("Mean without B2B:", round(y1[y1 <= 500].mean(), 2), round(y2[y2 <= 500].mean(), 2))
print("Truth:           ", round(truth[1], 2), round(truth[2], 2))

Median:           43.06 42.78
Trimmed mean:     48.53 48.51
Mean without B2B: 57.12 56.9
Truth:            57.83 57.07


| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median |Y1 43.06, Y2 42.78|We want the typical basket, not the average dollar value|When we need the average.|
| Trimmed mean |Y1 48.53, Y2 48.51 |The extreme values are errors |When large orders are real|
| Mean after B2B exclusion | Y1 57.12, Y2 56.90 | B2B orders are much bigger than consumer orders, so a 500 cutoff separates them | When some consumer orders are above $500 or some B2B orders are below it.|

**Which number should the dashboard show?** Commit to one and defend it.

The dashboard should show the mean after removing cancelled orders and orders above 500. The dashboard reports the average basket, so the statistic should still be a mean. The median and trimmed mean are far below the true average (43 and 49 vs 57–58).

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [57]:
import os
os.makedirs("src", exist_ok=True)

In [58]:
%%writefile src/basket_metrics.py
"""Tools to audit basket-value data and compute a robust average."""

import numpy as np
import pandas as pd
from scipy import stats

def audit_report(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: missingness, dtype, skew and outlier count.

    Outliers = values outside Q1 - 1.5*IQR and Q3 + 1.5*IQR.
    Skew and outliers are left empty for columns that are not numbers.
    """
    rows = []
    for col in df.columns:
        missing = df[col].isna().sum()
        dtype = str(df[col].dtype)
        skew = None
        outliers = None
        if dtype in ["int64", "float64"]:
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1
            skew = df[col].skew()
            outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
        rows.append([col, missing, dtype, skew, outliers])
    return pd.DataFrame(rows, columns=["column", "missing", "dtype", "skew", "outliers"]).set_index("column")


def robust_mean(x: pd.Series, method: str = "median") -> float:
    """A robust centre for x. method: "median", "trimmed" or your exclusion rule.

    "exclude" is my B2B rule from Phase 2: drop orders above $500, then take the mean.
    """
    x = pd.Series(x).dropna()
    if method == "median":
        return float(np.median(x))
    elif method == "trimmed":
        return float(stats.trim_mean(x, 0.1))
    elif method == "exclude":
        return float(x[x <= 500].mean())
    else:
        raise ValueError("method should be median, trimmed or exclude")


if __name__ == "__main__":
    # A short demonstration that runs when the file is run as a script
    orders = pd.Series([25, 38, 42, 45, 51, 60, 73, 90, 120, 1800])
    print("plain mean:", orders.mean())
    print("median:", robust_mean(orders, "median"))
    print("trimmed:", robust_mean(orders, "trimmed"))
    print("exclude:", robust_mean(orders, "exclude"))


Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [59]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)     # picks up your edits when you re-run %%writefile

# At least three properties that must hold, for example:
# assert abs(basket_metrics.robust_mean(symmetric_data) - symmetric_data.mean()) < tolerance
test_rng = np.random.default_rng(1)   # new data the module has not seen

symmetric = pd.Series(test_rng.normal(100, 10, 10000))
assert abs(basket_metrics.robust_mean(symmetric, "median") - symmetric.mean()) < 1
assert abs(basket_metrics.robust_mean(symmetric, "trimmed") - symmetric.mean()) < 1

baskets = pd.Series(test_rng.lognormal(np.log(45), 0.8, 1000))
with_huge = pd.concat([baskets, pd.Series([100000])])
assert with_huge.mean() - baskets.mean() > 50
assert abs(basket_metrics.robust_mean(with_huge, "trimmed") - basket_metrics.robust_mean(baskets, "trimmed")) < 1

consumer_orders = pd.Series([20, 40, 60])
b2b_orders = pd.Series([1500, 2000])
assert basket_metrics.robust_mean(pd.concat([consumer_orders, b2b_orders]), "exclude") == consumer_orders.mean()

test_df = pd.DataFrame({"basket_value": [10, None, 30, None]})
assert basket_metrics.audit_report(test_df).loc["basket_value", "missing"] == 2

print("All checks passed.")
print(basket_metrics.audit_report(dashboard))

All checks passed.
              missing    dtype       skew  outliers
column                                             
customer_id         0    int64   0.013578       0.0
year                0    int64  -0.083094       0.0
basket_value        0  float64  22.645259    1706.0
cancelled           0     bool        NaN       NaN


---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:**
https://github.com/zzzbky03/econ5200-ps1/blob/main/ai-appendix.md

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**

In [60]:
# The analysis that settles it
rows = []
for cutoff in [200, 300, 500, 750, 1000, 2000]:
    kept = no_cancel[no_cancel["basket_value"] <= cutoff]
    dropped = no_cancel[no_cancel["basket_value"] > cutoff]
    avg = kept.groupby("year")["basket_value"].mean()
    rows.append([cutoff, avg[1], avg[2], (avg[2] / avg[1] - 1) * 100,
                 (dropped["customer_id"] <= 8000).sum(),
                 (kept["customer_id"] > 8000).sum()])

sweep = pd.DataFrame(rows, columns=["cutoff", "year1", "year2", "change_%", "real_orders_dropped", "b2b_orders_kept"])
print(sweep.round(2))
print("Truth:", round(truth[1], 2), round(truth[2], 2), round(true_change, 2))
print("Dashboard change:", round(naive_change, 2))

   cutoff  year1  year2  change_%  real_orders_dropped  b2b_orders_kept
0     200  51.67  51.81      0.27                  599                0
1     300  55.10  55.07     -0.04                  167                1
2     500  57.12  56.90     -0.38                   23                5
3     750  58.48  57.19     -2.22                    1               20
4    1000  58.75  57.83     -1.57                    1               33
5    2000  60.55  61.22      1.11                    0               77
Truth: 57.83 57.07 -1.33
Dashboard change: 3.51


**What you found, including if it changes your Phase 2 recommendation:**

The objection is partly right. The cutoff changes the level: 200 drops 599 real orders, and 2,000 keeps 77 B2B orders. At 500 only 23 real orders get dropped and 5 B2B orders stay, so it's within 1 of the truth. But growth moves from −2.22% to +1.11% depending on the cutoff, so I can't trust one exact number. Still, no cutoff gets close to the dashboard's +3.51%.

So I changed my recommendation: keep the 500 mean for the level, but report growth as "flat (about −2% to +1%)". Later, B2B should be removed using a customer-type field instead of a fixed 500, which also fixes the inflation problem.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

## Average consumer basket stayed flat at about 57. It did not go up 3.5%.

**What was wrong:** In year 2 the dashboard started counting cancelled orders, and it also included big B2B orders. Cancelled orders caused most of the fake growth (4.1 of 4.8 points). I removed both and got 57.12 in year 1 and 56.90 in year 2.

**How sure:** Pretty sure there's no real growth. Any cutoff from 200 to 2,000 gives somewhere between −2% and +1%, nowhere near +3.5%.

**Still want:** A field that says if a customer is a consumer or a business, so I don't have to rely on a 500 cutoff.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors